# Amazon Bedrock curso - Modelos

Anotações do curso de Amazon Bedrock: como conectar o Python à AWS, chamar os modelos de fundação, ajustar os parâmetros de inferência, comparar modelos de fabricantes diferentes e trabalhar com imagens.

## Sumário

1. [Configuração do ambiente](#1.-Configuração-do-ambiente)
2. [Teste de conexão com a AWS (S3)](#2.-Teste-de-conexão-com-a-AWS-(S3))
3. [Conceitos fundamentais](#3.-Conceitos-fundamentais)
4. [Conectando ao Amazon Bedrock](#4.-Conectando-ao-Amazon-Bedrock)
5. [Primeira inferência com `invoke_model`](#5.-Primeira-inferência-com-invoke_model)
6. [Testando os parâmetros](#6.-Testando-os-parâmetros)
7. [Comparando modelos da família Nova](#7.-Comparando-modelos-da-família-Nova)
8. [Modelos de outros fabricantes e o método `converse`](#8.-Modelos-de-outros-fabricantes-e-o-método-converse)
9. [Modelos multimodais (imagem + texto)](#9.-Modelos-multimodais-(imagem-+-texto))
10. [Resumo dos aprendizados](#10.-Resumo-dos-aprendizados)

---
## 1. Configuração do ambiente

O nosso objetivo agora é fazer a conexão desse arquivo com a AWS, utilizando as chaves criadas anteriormente, e depois fazer as chamadas aos modelos.

Isso é feito utilizando uma biblioteca chamada **Boto3**: é ela que faz a conexão da nossa aplicação em Python com a AWS.

In [1]:
# Instalando o Boto3, biblioteca que faz a conexão da nossa aplicação em Python com a AWS.
# O %pip instala no mesmo ambiente do kernel do notebook (mais seguro que !pip).
%pip install boto3

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1.1 Credenciais da AWS

> ⚠️ **Boa prática de segurança:** nunca escreva a `aws_access_key_id` e a `aws_secret_access_key` direto no código. Qualquer pessoa que tiver acesso ao notebook (GitHub, e-mail, print de tela) passa a ter acesso à sua conta da AWS — e às cobranças.

Por isso as chaves são lidas do arquivo `curso_bedrock_accessKeys.csv` (baixado no console do IAM ao criar a chave), que fica fora do código. Criamos **uma única sessão** (`boto3.Session`) com essas credenciais e a região, e todos os serviços (S3, Bedrock e Bedrock Runtime) saem dela — assim a configuração fica em um só lugar.

Alternativa ainda melhor: rodar `aws configure` no terminal uma vez. Aí basta `boto3.Session(region_name="us-east-1")`, sem informar chave nenhuma.

In [ ]:
# Importando a biblioteca boto3 para interagir com os serviços da AWS
import csv
import boto3

# Lendo as credenciais do arquivo CSV gerado pelo IAM (fora do código)
with open("curso_bedrock_accessKeys.csv", encoding="utf-8-sig") as f:
    credenciais = next(csv.DictReader(f))

# Sessão única: todos os serviços da AWS são criados a partir dela
session = boto3.Session(
    aws_access_key_id=credenciais["Access key ID"],
    aws_secret_access_key=credenciais["Secret access key"],
    region_name="us-east-1",
)

---
## 2. Teste de conexão com a AWS (S3)

No nosso primeiro exemplo, vamos fazer um teste de conexão com a AWS.

Com o Boto3 é possível fazer a conexão com todos os serviços disponíveis que são aplicáveis. Vou fazer uma conexão com o **S3**, que é o serviço de armazenamento, só para entendermos melhor o funcionamento.

In [2]:
# Testando a conexão com o Amazon S3 usando a sessão criada com as credenciais
s3 = session.resource('s3')

Agora vamos percorrer todos os buckets (as "pastas" armazenadas no S3) e mostrar o nome de cada um:

- Acessamos o **`s3.buckets.all()`**, que lista todos os buckets;
- Fazemos um print em **`bucket.name`**.

Se a conta não tiver nenhum bucket, o resultado sai vazio — e tudo bem.

In [3]:
# Listando todos os buckets disponíveis na conta da AWS
for bucket in s3.buckets.all():
  print(bucket.name)

bia-assets-471116065639
codepipeline-us-east-1-acd194f489e7-4a30-9020-8941ed8a2dfb


✅ Note que não foi gerada nenhuma mensagem de erro, o que indica que a conexão foi feita corretamente.

---
## 3. Conceitos fundamentais

Antes de chamar os modelos, vale entender os parâmetros e termos que aparecem o tempo todo no Bedrock.

### 3.1 Resumo dos parâmetros

- **Max Tokens:**
    - Define o limite máximo de tokens que o modelo pode gerar na resposta.
    - Esse parâmetro ajuda a controlar o tamanho da saída e pode contribuir para a redução de custos, já que muitos modelos cobram pelo volume de tokens gerados.
    - É importante lembrar que o limite não obriga o modelo a utilizar todos os tokens disponíveis. Ele pode finalizar a resposta antes de atingir esse máximo, caso considere que já respondeu à solicitação.
    - Exemplo: se o limite for de 500 tokens, o modelo poderá gerar uma resposta com 200 tokens, mas não poderá ultrapassar o limite configurado.
- **Temperature:**
    - Controla o grau de aleatoriedade na seleção dos próximos tokens durante a geração da resposta.
        - Temperatura baixa: tende a produzir respostas mais previsíveis e consistentes.
        - Temperatura alta: tende a produzir respostas mais variadas e menos previsíveis.
    - A temperatura não controla diretamente a inteligência ou a criatividade do modelo. Ela altera a distribuição de probabilidades utilizada na geração.
    - Exemplo: para extrair informações de documentos ou gerar respostas padronizadas, uma temperatura baixa costuma ser útil. Para brainstorming, uma temperatura mais alta pode favorecer a variedade.
- **Top P:**
    - Controla a seleção dos tokens candidatos com base na soma de suas probabilidades, utilizando uma técnica conhecida como nucleus sampling.
    - O modelo considera os tokens mais prováveis até que a soma acumulada das probabilidades atinja o valor definido por Top P.
        - Top P baixo: restringe a seleção a um conjunto menor de tokens mais prováveis.
        - Top P alto: permite considerar um conjunto maior de tokens candidatos.
    - Diferença entre Temperature e Top P: a temperatura modifica a distribuição de probabilidades dos tokens; o Top P restringe o conjunto de candidatos considerados na amostragem.
    - Exemplo: imagine que o modelo possa escolher entre diversas palavras para completar uma frase. O Top P limita o conjunto de candidatas com base em suas probabilidades, enquanto a temperatura influencia como essas probabilidades são distribuídas.
- **Stop Sequences:**
    - Permitem definir sequências de caracteres que, quando reconhecidas durante a geração, fazem o modelo interromper a resposta.
    - Essas sequências podem ser palavras, símbolos ou combinações de caracteres, dependendo das regras de cada modelo.
    - Exemplo: se `###FIM###` for configurada como sequência de parada e o modelo a gerar, a geração será interrompida nesse ponto. A sequência pode ou não aparecer na resposta retornada, conforme o comportamento do modelo.
    - Esse recurso pode ser útil para delimitar respostas, separar seções ou controlar formatos de saída. Em agentes, também pode auxiliar na estruturação de determinados fluxos, mas não substitui mecanismos de controle de execução, ferramentas ou regras de orquestração.
- **Context Window (janela de contexto):**
    - Representa a quantidade máxima de tokens que um modelo consegue processar como contexto em uma execução, respeitando os limites específicos de sua arquitetura e implementação.
    - Dependendo do modelo e da API utilizada, esse limite pode envolver os tokens de entrada e de saída, além de outros componentes considerados pela implementação.
    - A entrada pode incluir instruções de sistema, prompt do usuário, histórico da conversa e conteúdo recuperado de documentos. Em aplicações com agentes, também pode incluir resultados de ferramentas.
    - Exemplo: em uma conversa longa, o histórico acumulado pode consumir boa parte da janela de contexto, deixando menos espaço disponível para novos dados e para a resposta.
    - Atenção: janela de contexto e Max Tokens são conceitos diferentes. A primeira limita o contexto total suportado; o segundo limita a quantidade máxima de tokens gerados na saída. Os limites exatos dependem do modelo.
- **Streaming:**
    - É um mecanismo de entrega progressiva da resposta, no qual os tokens gerados pelo modelo são enviados ao cliente à medida que ficam disponíveis, em vez de aguardar a conclusão de toda a resposta.
    - Isso melhora a percepção de velocidade e permite que o usuário comece a ler antes de a geração terminar.
    - Exemplo: em um chatbot, o texto aparece gradualmente na tela enquanto o modelo ainda está produzindo a resposta.
    - Importante: streaming é um mecanismo de transmissão da resposta, não um parâmetro de criatividade nem uma modalidade de entrada. O suporte depende da API e do modelo utilizado.
- **Multimodalidade:**
    - É a capacidade de um modelo processar ou gerar diferentes tipos de dados, como texto, imagens, áudio e vídeo, conforme as modalidades suportadas pelo modelo.
    - É possível classificar as tarefas de acordo com os dados de entrada e de saída:
        - **Text-to-Text** → Recebe texto e gera texto.
        - **Image-to-Text** → Analisa uma imagem e gera texto, por exemplo, uma descrição.
        - **Video-to-Text** → Analisa um vídeo e gera texto, como um resumo de suas cenas.
        - **Speech-to-Text** → Converte fala em texto.
        - **Text-to-Speech** → Converte texto em fala.
        - **Audio-to-Audio** → Recebe áudio e gera áudio, conforme o modelo.
    - Um modelo multimodal não necessariamente suporta todas essas modalidades. É preciso verificar as capacidades específicas do modelo disponibilizado no Amazon Bedrock.

> 💡 Os parâmetros **Max Tokens** e **Temperature** já resolvem muitos problemas.

Esses são os parâmetros e conceitos mais importantes que encontramos ao trabalhar no Bedrock, e compreendê-los ajuda a obter respostas melhores e a construir aplicações mais eficientes, escaláveis e econômicas.

### 3.2 Glossário rápido

| Termo | O que é |
| --- | --- |
| **Token** | Pedaço de palavra. É a unidade que a AWS conta e cobra. Em português, ~1 token a cada 3 ou 4 letras. |
| **Prompt** | A instrução que enviamos ao modelo. |
| **Inferência** | Uma chamada ao modelo: mandamos o prompt, ele devolve a resposta. É o que se paga. |
| **Foundation model** | Modelo já treinado e pronto para uso, oferecido pelo Bedrock. Não treinamos nada. |
| **Modalidade** | Tipo de entrada ou saída aceita: `TEXT`, `IMAGE`, `VIDEO`, `SPEECH`, `EMBEDDING`. |
| **Runtime** | O serviço que de fato executa a inferência (`bedrock-runtime`), em oposição ao serviço de catálogo e administração (`bedrock`). |
| **On-demand** | Paga-se por token usado, sem reserva. É o nosso caso. |
| **Inference profile** | Roteamento entre regiões. Alguns modelos só funcionam assim e exigem o prefixo `us.` no ID. |

---
## 4. Conectando ao Amazon Bedrock

O Bedrock é dividido em **dois serviços**, e cada um tem seu próprio cliente:

| `service_name` | Para que serve | Exemplos de uso |
| --- | --- | --- |
| **`'bedrock'`** | Catálogo e administração: informações básicas sobre a plataforma e os modelos de fundação disponíveis. | `list_foundation_models()` |
| **`'bedrock-runtime'`** | Execução: é ele que efetivamente aplica os modelos nas nossas aplicações de IA generativa (inferência). | `invoke_model()`, `converse()` |

Repare que aqui usamos **`boto3.client`**, e no S3 usamos **`boto3.resource`**.

- **`resource`** é uma interface de alto nível, orientada a objetos. Por isso escrevemos `s3.buckets.all()`, que parece Python comum.
- **`client`** é a interface de baixo nível, que espelha a API da AWS chamada por chamada.

O Bedrock **só tem `client`** — então é sempre assim que vamos acessá-lo.

### 4.1 Cliente `bedrock` — catálogo de modelos

In [4]:
# Configurando o cliente do Amazon Bedrock (catálogo e administração) a partir da sessão
bedrock = session.client(service_name='bedrock')

Com o cliente pronto, a primeira coisa útil é perguntar ao Bedrock **quais modelos existem** na região em que estamos conectados.

In [5]:
# Listar os modelos de fundação disponíveis no Amazon Bedrock que podemos utilizar para criar aplicações de IA generativa.
bedrock.list_foundation_models()

{'ResponseMetadata': {'RequestId': '25a204a9-e7fe-44be-9f58-bcbde51bb710',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 27 Oct 2026 12:16:44 GMT',
   'content-type': 'application/json',
   'content-length': '212764',
   'connection': 'keep-alive',
   'x-amzn-requestid': '25a204a9-e7fe-44be-9f58-bcbde51bb710'},
  'RetryAttempts': 0},
 'modelSummaries': [{'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/openai.gpt-6-astra',
   'modelId': 'openai.gpt-6-astra',
   'modelName': 'GPT-6 Astra',
   'providerName': 'OpenAI',
   'inputModalities': ['TEXT', 'IMAGE'],
   'outputModalities': ['TEXT'],
   'responseStreamingSupported': True,
   'customizationsSupported': [],
   'inferenceTypesSupported': ['INFERENCE_PROFILE'],
   'modelLifecycle': {'status': 'ACTIVE',
    'startOfLifeTime': datetime.datetime(2026, 9, 8, 17, 0, tzinfo=tzutc())}},
  {'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/nvidia.nemotron-nano-12b-v2',
   'modelId': 'nvidia.nemotron-nano-12b-v2',
  

A saída é longa, mas só quatro campos de cada modelo interessam no dia a dia:

- **`modelId`** — o identificador que vamos passar nas chamadas.
- **`inputModalities`** — o que o modelo aceita receber. Se não tiver `IMAGE`, não adianta mandar imagem.
- **`inferenceTypesSupported`** — `ON_DEMAND` significa que podemos chamar pelo `modelId` direto. Se aparecer **só** `INFERENCE_PROFILE`, o ID puro é recusado e precisamos do prefixo `us.`.
- **`modelLifecycle`** — `ACTIVE`, `LEGACY` (já tem data para sair do ar) ou `DEPRECATED`.

⚠️ Aparecer nessa lista não quer dizer que temos acesso. O acesso é liberado por região, no console, em `Bedrock → Model access`.

### 4.2 Cliente `bedrock-runtime` — execução dos modelos

Para realmente aplicar os modelos de fundação em nossas aplicações de IA generativa, precisamos do cliente **`'bedrock-runtime'`**.

In [ ]:
# Configurando o cliente do Amazon Bedrock Runtime: é ele que efetivamente executa os modelos.
bedrock_runtime = session.client(service_name='bedrock-runtime')

---
## 5. Primeira inferência com `invoke_model`

### 5.1 Escolhendo o modelo

Vamos usar o modelo **`amazon.nova-lite-v1:0`**, porque tem uma boa qualidade, custo reduzido e é ideal para demonstrações.

O `MODEL_ID` não é um nome solto — ele tem estrutura. Em `amazon.nova-lite-v1:0`:

| Parte | Significado |
| --- | --- |
| `amazon` | fabricante do modelo |
| `nova-lite` | nome e porte do modelo |
| `v1` | versão |
| `:0` | revisão dessa versão |

Guardamos numa constante em maiúsculas porque a função `gerar_resposta` vai lê-la como variável global. Para trocar de modelo, basta alterar aqui e reexecutar as células seguintes.

In [7]:
# Especificando o ID do modelo que queremos utilizar para gerar texto.
MODEL_ID = "amazon.nova-lite-v1:0"

### 5.2 Montando a requisição em JSON

A conversa com o Bedrock é toda em **JSON**, então precisamos da biblioteca para traduzir nos dois sentidos:

- **`json.dumps`** — dicionário Python → texto JSON. É o que sai daqui.
- **`json.loads`** — texto JSON → dicionário Python. É o que chega de volta.

In [8]:
# Importando a biblioteca json para manipular dados em formato JSON
import json

Para evitar a repetição de código, vamos criar uma função responsável por enviar prompts para o modelo.

In [9]:
# Definir o formato do json que é requisitado pelo modelo para gerar texto.
def gerar_resposta(prompt, temperature=0.7, top_p=0.9, max_tokens=50):
    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"text": prompt}
                ]
            }
        ],
        "inferenceConfig": {
            "temperature": temperature,
            "topP": top_p,
            "maxTokens": max_tokens
        }
    }
    # Fazendo a chamada para o modelo. json.dumps(body) converte o dicionário Python em uma string JSON.
    response = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(body))
    # Lendo a resposta do modelo (stream de bytes) e convertendo para dicionário Python.
    response_body = json.loads(response["body"].read())
    # Retornando o texto gerado pelo modelo.
    return response_body["output"]["message"]["content"][0]["text"]

Precisamos dessa estrutura para enviar uma mensagem para o modelo.

- `"role"`: indica que a mensagem é do usuário.
- `"content"`: conteúdo do usuário. É uma **lista**, porque uma mensagem pode ter texto e imagem juntos.
- `"inferenceConfig"`: configuração de inferência (`temperature`, `topP`, `maxTokens`).

**E o caminho de volta?** O `invoke_model` não devolve texto pronto. Ele devolve um *stream*, e por isso precisamos de três passos:

1. `response["body"].read()` — lê os bytes do stream.
2. `json.loads(...)` — converte esses bytes em dicionário Python.
3. `["output"]["message"]["content"][0]["text"]` — desce até o texto gerado.

Esse caminho é o trecho que mais custa relembrar depois. O `[0]` existe porque `content` é lista: pegamos o primeiro bloco.

### 5.3 Testando a função

Primeira inferência de verdade. O prompt vai entre três aspas para podermos escrever várias linhas.

Como não passamos nenhum parâmetro além do prompt, valem os valores padrão definidos na função: `temperature=0.7`, `top_p=0.9` e `max_tokens=50`.

In [10]:
# Fazendo o teste enviando o prompt como parâmetro
prompt = """
Explique o que é Machine Learning.
"""
# Fazendo a chamada para o modelo de IA generativa do Amazon Bedrock e imprimindo a resposta gerada.
print(gerar_resposta(prompt))

Machine Learning (ML) é uma subdisciplina da Inteligência Artificial (IA) que se concentra na capacidade de sistemas computacionais aprenderem e melhorarem automaticamente a partir da experiência, sem serem explicitamente programados. Em vez de seguir um


---
## 6. Testando os parâmetros

Nesta seção vemos, na prática, o efeito de cada parâmetro descrito na [seção 3.1](#3.1-Resumo-dos-parâmetros).

### 6.1 `max_tokens` — tamanho da resposta

É o **teto da resposta**, não da pergunta. O modelo para de escrever ao atingi-lo, mesmo no meio de uma frase.

- Valores úteis: 50 para respostas curtas, 300 a 500 para explicações.
- Compare esta célula (`50`) com a próxima (`500`): a primeira corta no meio.

⚠️ Resposta maior custa mais, porque paga-se por token gerado. E há um detalhe de cota: a AWS debita `tokens de entrada + max_tokens` **no início** da requisição, devolvendo o que sobrar só no final. Pedir `max_tokens` muito alto "reserva" cota que talvez nem seja usada.

In [11]:
# Testando a função gerar_resposta com diferentes parâmetros
print(
    gerar_resposta(
        prompt,
        max_tokens=50
    )
)

Machine Learning (ML) é uma subárea da Inteligência Artificial (IA) que se concentra na capacidade de sistemas computacionais aprenderem e melhorarem a partir da experiência, sem serem explicitamente programados para realizar tarefas específicas.


In [12]:
# Testando a função gerar_resposta com max_tokens maior, para ver se a resposta é mais completa.
# Esse é um custo maior, pois o modelo vai gerar mais tokens.
print(
    gerar_resposta(
        prompt,
        max_tokens=500
    )
)

Machine Learning (ML) é uma área da Inteligência Artificial que se concentra na capacidade dos sistemas computacionais de aprender e melhorar automaticamente a partir da experiência, sem serem explicitamente programados para realizar tarefas específicas. Em vez de seguir um conjunto predefinido de regras, os algoritmos de Machine Learning analisam dados, identificam padrões e fazem previsões ou decisões com base nesses padrões.

Aqui estão alguns conceitos-chave relacionados ao Machine Learning:

1. **Tipos de Machine Learning:**
   - **Aprendizado Supervisionado:** O modelo é treinado com dados rotulados, onde a saída desejada é conhecida. Exemplos incluem classificação (como spam vs. não spam em e-mails) e regressão (como prever preços de imóveis).
   - **Aprendizado Não Supervisionado:** O modelo é treinado com dados não rotulados. O objetivo é encontrar estruturas ou padrões nos dados. Exemplos incluem clusterização (como agrupar clientes em segmentos) e associação (como análise de

> **Nota:** o tempo de execução e o custo da inferência na segunda chamada são maiores que na primeira.

### 6.2 `temperature` — "criatividade"

Controla o quanto o modelo se arrisca ao escolher a próxima palavra. (Tecnicamente ela não controla a criatividade em si, e sim a distribuição de probabilidades — ver [seção 3.1](#3.1-Resumo-dos-parâmetros) —, mas na prática o efeito percebido é esse.)

- **Perto de 0** → previsível e repetível. A mesma pergunta tende a dar a mesma resposta.
- **Perto de 1** → criativo e variado, com mais risco de divagar.

Compare o resultado desta célula (`0.1`) com o da próxima (`1.0`) usando o mesmo prompt de slogan.

In [13]:
# Teste com um outro prompt
prompt = """
Crie um slogan para uma escola de Inteligência Artificial.
"""

In [ ]:
# Com temperature baixa, a resposta tende a ser menos criativa e mais objetiva e direta.
print(
    gerar_resposta(
        prompt,
        temperature=0.1
    )
)

"Despertando mentes para o futuro da Inteligência Artificial." 

Este slogan busca transmitir a ideia de que a escola não apenas ensina habilidades técnicas, mas também inspira e prepara os alunos para os desafios e oportunidades do mundo impulsion


In [ ]:
# Com temperature mais alta o modelo tende a dar respostas mais criativas, mas também mais arriscadas,
# pois pode gerar respostas que não fazem sentido.
print(
    gerar_resposta(
        prompt,
        temperature=1.0
    )
)

"Despertando mentes para o futuro da Inteligência Artificial." 

Este slogan sugere que a escola está não apenas ensinando habilidades técnicas, mas também inspirando e preparando os estudantes para os desafios e oportunidades que o futuro


### 6.3 `top_p` — diversidade do vocabulário

Enquanto a `temperature` mexe no risco da escolha, o `top_p` define **de quantas palavras candidatas** o modelo pode escolher. Com `0.3` ele fica restrito às mais prováveis; com `1.0` o leque inteiro está liberado.

Os dois parâmetros agem juntos. Na prática costuma-se ajustar um e deixar o outro fixo — mexer nos dois ao mesmo tempo dificulta entender de onde veio a mudança.

In [16]:
# top_p baixo: o modelo escolhe só entre as palavras mais prováveis.
print(
    gerar_resposta(
        prompt,
        temperature=0.7,
        top_p=0.3
    )
)

"Despertando mentes para o futuro: Maestria em Inteligência Artificial." 

Este slogan busca capturar a essência de uma escola dedicada à IA, enfatizando o desenvolvimento de habilidades e conhecimentos que preparam os alunos para


In [17]:
# top_p = 1.0: todo o leque de palavras candidatas está liberado.
print(
    gerar_resposta(
        prompt,
        temperature=0.7,
        top_p=1.0
    )
)

"Despertando mentes para o futuro: Maestria em Inteligência Artificial." 

Este slogan busca enfatizar a ideia de que a escola não apenas ensina habilidades técnicas, mas também inspira e prepara os alunos para os desafios e oportunidades


### 6.4 Mesmos parâmetros, prompts diferentes

Depois de testar algumas inferências, vamos ver como a escolha da `temperature` depende do **tipo de tarefa** do prompt.

#### Tarefa criativa → temperatura alta

In [18]:
prompt = """
Crie o nome de uma startup de Inteligência Artificial.
"""

Aqui está a demonstração prática da `temperature` alta: a **mesma** pergunta, cinco vezes seguidas, com `temperature=1.0`. Cada resposta sai diferente, esse comportamento é esperado em modelos generativos.

⚠️ São cinco inferências faturadas em uma única execução.

In [19]:
# Mesma pergunta 5 vezes com temperature alta: cada resposta sai diferente.
for i in range(5):
    print("=" * 50)
    print(gerar_resposta(prompt, temperature=1.0))

Claro! Aqui estão algumas sugestões para o nome de uma startup de Inteligência Artificial:

1. **AetherAI**
2. **NeuroNexus**
3. **InsigAI**
4. **CognoSphere**
5
Claro! Aqui estão algumas sugestões para o nome de uma startup de Inteligência Artificial:

1. **NeuroSpark AI**
2. **Cogito Labs**
3. **AetherMind AI**
4. **InsightX Labs**

Claro! Aqui está uma sugestão para o nome de uma startup de Inteligência Artificial:

**NeuroGenius AI**

Esta nome combina a ideia de inteligência ("Genius") com elementos relacionados à ciência neurológica e computacional ("Neuro
Claro! Aqui estão algumas sugestões para nomes de startups de Inteligência Artificial:

1. **NeuroInno**
2. **AetherAI**
3. **Cognita Labs**
4. **VitaMind AI**
5
Claro! Aqui está uma sugestão para o nome de uma startup de Inteligência Artificial:

**NeuroInno**

Esta combinação de "Neuro", referindo-se a neurociência e redes neurais, e "Inno", que significa


#### Tarefa técnica → temperatura baixa

In [20]:
prompt = """
Explique a diferença entre regressão linear e regressão logística.
"""

Pergunta técnica pede `temperature` **baixa**. Com `0.2` o modelo fica conservador e se atém ao que é consensual — o oposto do que queríamos no slogan.

Regra prática: **criação** pede temperatura alta; **explicação e extração de dados** pedem temperatura baixa.

In [ ]:
# Para conteúdos técnicos normalmente utilizamos temperaturas menores, porque buscamos consistência e precisão, e não criatividade.
print(
    gerar_resposta(
        prompt,
        temperature=0.2,
        max_tokens=400
    )
)

A regressão linear e a regressão logística são duas técnicas estatísticas amplamente utilizadas na análise de dados, mas servem a propósitos diferentes e são aplicáveis em contextos distintos. Aqui estão as principais diferenças entre elas:

### Regressão Linear

**Objetivo:**
A regressão linear é usada para modelar a relação entre uma variável dependente contínua e uma ou mais variáveis independentes. O objetivo é prever valores numéricos contínuos.

**Tipo de Variável Dependende:**
A variável dependente na regressão linear é contínua, o que significa que ela pode assumir qualquer valor dentro de um intervalo, como altura, peso, temperatura, etc.

**Modelo:**
O modelo da regressão linear é geralmente expresso como:
\[ y = \beta_0 + \beta_1x_1 + \beta_2x_2 +... + \beta_nx_n + \epsilon \]
onde \( y \) é a variável dependente, \( \beta_0 \) é a interceptação, \( \beta_1, \beta_2,..., \beta_n \) são os coeficientes das variáveis independentes \( x_1, x_2,..., x_n \), e \( \epsilon \) é o 

---
## 7. Comparando modelos da família Nova

Nos testes anteriores fizemos inferências com um único modelo. Agora vamos explorar um dos maiores benefícios do Bedrock: **utilizar modelos diferentes por meio da mesma interface** — o código não muda, o que muda é o identificador do modelo.

Os três portes da família Nova, do menor para o maior:

| Modelo | Perfil |
| --- | --- |
| **Nova Micro** | Só texto. O mais rápido e barato. Bom para classificar e resumir. |
| **Nova Lite** | Multimodal. Equilíbrio entre custo e qualidade. |
| **Nova Pro** | Multimodal. O mais capaz da família, mais caro e mais lento. |

Guardamos num dicionário `{"nome amigável": "model_id"}` para poder percorrer todos num laço.

In [22]:
MODELS = {
    "Nova Micro": "amazon.nova-micro-v1:0",
    "Nova Lite": "amazon.nova-lite-v1:0",
    "Nova Pro": "amazon.nova-pro-v1:0"
}

Esta função é quase idêntica à `gerar_resposta`, com **uma diferença**: o `model_id` chega como parâmetro em vez de vir da constante global `MODEL_ID`.

É essa mudança que permite rodar o mesmo prompt em vários modelos dentro de um laço.

In [23]:
def gerar_resposta_modelo(model_id, prompt, temperature=0.7, top_p=0.9, max_tokens=300):
    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"text": prompt}
                ]
            }
        ],
        "inferenceConfig": {
            "temperature": temperature,
            "topP": top_p,
            "maxTokens": max_tokens
        }
    }

    response = bedrock_runtime.invoke_model(modelId=model_id, body=json.dumps(body))
    response_body = json.loads(response["body"].read())
    return response_body["output"]["message"]["content"][0]["text"]

### 7.1 Comparando a qualidade das respostas

In [24]:
prompt = """
Explique computação quântica para uma criança de 10 anos.
"""

Agora o laço: para cada item do dicionário `MODELS`, chamamos a mesma função mudando só o `model_id`.

Assim conseguimos ler as três respostas em sequência e comparar o estilo e a profundidade de cada modelo diante do mesmo prompt.

In [25]:
for nome, model_id in MODELS.items():
    print("=" * 80)
    print(nome)
    print("=" * 80)

    resposta = gerar_resposta_modelo(model_id, prompt)

    print(resposta)
    print('\n')

Nova Micro
Claro! Vamos imaginar que você tem um jogo de memória com várias cartas. Em cada carta, você tem uma informação, como uma cor ou um número. Para encontrar o par de cartas que combina, você precisa virá-las uma por uma e ver se elas combinam.

Agora, imagine se você pudesse virar várias cartas ao mesmo tempo, sem precisar tocá-las individualmente. Isso é um pouco como a computação quântica. 

Na computação quântica, em vez de usarmos simples "0" ou "1" (como na computação normal), usamos algo chamado de "qubits". Um qubit pode ser um 0, um 1, ou até mesmo ambos ao mesmo tempo! Isso é chamado de "suspensão quântica" ou "superposição". E isso permite que a computação quântica faça muitas coisas muito mais rápido do que computadores normais.

Também há algo chamado de "entrelaçamento quântico", que é um pouco como se duas cartas no seu jogo de memória pudessem saber sobre o que a outra carta está sem olhar diretamente uma para a outra. Isso ajuda os computadores quânticos a trab

**Essa é uma das grandes vantagens de usar o Amazon Bedrock**: com o mesmo código podemos utilizar modelos diferentes.

### 7.2 Comparando o tempo de execução

Além da qualidade da resposta, o tempo de execução também diferencia os modelos.

In [26]:
import time

Marcamos o instante antes e depois da chamada e mostramos a diferença. Note que descartamos a resposta — aqui só interessa o relógio.

O resultado confirma o esperado: **Micro < Lite < Pro** (na execução registrada: 1,78 s, 2,64 s e 3,44 s). Modelo maior pensa melhor e demora mais.

In [27]:
for nome, model_id in MODELS.items():
    inicio = time.time()

    gerar_resposta_modelo(model_id, prompt)  # a resposta é descartada: só interessa o tempo

    fim = time.time()

    print(f"{nome}: {fim - inicio:.2f} segundos")

Nova Micro: 1.78 segundos
Nova Lite: 2.64 segundos
Nova Pro: 3.44 segundos


---
## 8. Modelos de outros fabricantes e o método `converse`

### 8.1 O problema: cada fabricante tem seu próprio JSON

Agora vamos testar com um modelo que não faz parte da família **Nova**, usando a mesma função de antes.

> ⚠️ A célula abaixo **gera um erro de propósito** (`ValidationException`).

In [28]:
gerar_resposta_modelo("deepseek.v3.2", prompt)

ValidationException: An error occurred (ValidationException) when calling the InvokeModel operation: Failed to deserialize the JSON body into the target type: ?[0]: Invalid 'messages': Invalid 'content': value did not match any expected variant at line 1 column 118

**Por que deu erro?** A mensagem acontece porque o **json** que montamos serve somente para a família de modelos **Nova**. Para modelos de outras famílias precisamos usar outro formato de json.

O motivo: o `invoke_model` exige o json **nativo de cada fabricante**. O corpo com `messages` + `inferenceConfig` é o padrão da Amazon Nova, e a DeepSeek simplesmente não reconhece esse desenho.

### 8.2 A solução: o método `converse`

Primeiro, um novo dicionário com modelos de **três fabricantes diferentes** (Amazon, DeepSeek e Qwen) — seriam quatro com o Claude, da Anthropic.

⚠️ Repare na linha do **Claude comentada**. Ela falha se for usada assim: na listagem de modelos, o Claude Sonnet aparece com `inferenceTypesSupported: ['INFERENCE_PROFILE']`, sem `ON_DEMAND`. Modelos assim não aceitam o `modelId` puro — precisam do prefixo da região: `us.anthropic.claude-sonnet-4-6`. O mesmo vale para GPT, Grok e outros da lista.

In [29]:
# Testando a função gerar_resposta_modelo com diferentes modelos de IA generativa do Amazon Bedrock
MODELS = {
    "Nova Lite": "amazon.nova-lite-v1:0",
    "Nova Pro": "amazon.nova-pro-v1:0",
    #"Claude Sonnet": "anthropic.claude-sonnet-4-6",
    "DeepSeek": "deepseek.v3.2",
    "Qwen": "qwen.qwen3-next-80b-a3b"
}

Aqui trocamos o `invoke_model` pelo **`converse`**. A diferença está em quem monta o json: no `invoke_model` somos nós; no `converse` é a própria API.

Repare que os parâmetros saem de dentro do `body` e viram argumentos diretos da chamada — `modelId`, `messages` e `inferenceConfig`. A leitura da resposta também fica mais curta: não há stream para ler, o dicionário já vem pronto.

In [30]:
# Criando a função conversar, que utiliza o método converse do cliente bedrock_runtime
# para interagir com qualquer modelo de IA generativa do Amazon Bedrock.
def conversar(model_id, prompt):
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[
            {
                "role": "user",
                "content": [
                    {"text": prompt}
                ]
            }
        ],
        inferenceConfig={
            "maxTokens": 300,
            "temperature": 0.7,
            "topP": 0.9
        }
    )

    return response["output"]["message"]["content"][0]["text"]

Quando usamos o método **invoke_model** precisamos enviar os dados exatamente no padrão do fabricante. Já com o método **converse** a própria API faz a conversão para o padrão de cada fabricante.

Na prática: a mesma função passa a servir para qualquer modelo do catálogo — é exatamente o que as próximas células demonstram, rodando Nova, DeepSeek e Qwen sem trocar uma linha sequer.

| | `invoke_model` | `converse` |
| --- | --- | --- |
| Quem monta o JSON | Nós, no formato nativo de cada fabricante | A própria API |
| Troca de fabricante | Exige reescrever o `body` | Basta trocar o `modelId` |
| Leitura da resposta | `json.loads(response["body"].read())` + caminho até o texto | `response["output"]["message"]["content"][0]["text"]` direto |
| Parâmetros | Dentro do `body` | Argumentos diretos (`messages`, `inferenceConfig`) |

O mesmo laço de antes, agora com três fabricantes — e **sem nenhum ajuste de formato** para cada um deles.

É esse o ganho concreto do `converse`: trocar de modelo passa a ser trocar uma string.

In [31]:
for nome, model_id in MODELS.items():
    print("=" * 80)
    print(nome)
    print("=" * 80)
    resposta = conversar(model_id, prompt)
    print(resposta)
    print('\n')

Nova Lite
Claro! Vamos imaginar que estamos falando sobre computação quântica de uma forma bem simples.

### O Computador Clássico
Primeiro, vamos pensar no computador que você usa todos os dias. Ele tem **bits**, que são como pequenas lâmpadas que podem estar **ligadas** (1) ou **desligadas** (0). Você pode pensar nisso como se fossem pequenos interruptores que podem estar em duas posições.

### O Computador Quântico
Agora, vamos falar sobre o computador quântico, que é muito diferente. Em vez de usar bits, ele usa **qubits**. Os qubits são como esses interruptores, mas com uma mágica especial: eles podem estar **ligados, desligados ou em qualquer lugar intermediário ao mesmo tempo!**

### A Magia dos Qubits
Vamos fazer uma analogia com a luz. Imagine que a luz pode ser como uma onda que pode ser clara ou escura. Um qubit pode ser como um feixe de luz que pode ser ajustado para ser qualquer mistura de claro e escuro ao mesmo tempo.

### Superposição
Isso é chamado de **superposição**.

---
## 9. Modelos multimodais (imagem + texto)

O objetivo é enviar uma imagem e pedir para o modelo analisá-la. A partir daqui saímos do texto puro: o modelo vai receber uma **imagem** junto com o prompt.

### 9.1 Lendo a imagem

Primeiro guardamos o nome do arquivo, que está na mesma pasta do notebook.

In [32]:
# Leitura da imagem local
arquivo = 'imagem_ia_teste.jpg'

In [33]:
arquivo

'imagem_ia_teste.jpg'

O `rb` significa *read binary*. Imagem não é texto, então lemos o arquivo como **bytes** — é esse conteúdo que o modelo espera receber, e não o caminho do arquivo.

In [34]:
# Abrindo o arquivo de imagem em modo binário e lendo seu conteúdo em bytes.
with open(arquivo, 'rb') as f:
    imagem_bytes = f.read()

### 9.2 Escolhendo um modelo que aceita imagem

Trocamos o modelo para o **Nova Pro**, que aceita `TEXT` e `IMAGE` na entrada.

⚠️ Nem todo modelo enxerga imagem. O **Nova Micro é só texto** — se mandarmos imagem para ele, a chamada falha. Para conferir antes, basta olhar o `inputModalities` na listagem de modelos ([seção 4.1](#4.1-Cliente-bedrock-—-catálogo-de-modelos)).

> **Atenção:** como `MODEL_ID` é uma variável global, a partir daqui a função `gerar_resposta` da seção 5 também passa a usar o Nova Pro.

In [35]:
# Esse modelo recebe texto e imagens
MODEL_ID = "amazon.nova-pro-v1:0"

### 9.3 Função para analisar imagens

O bloco de imagem entra dentro de `content`, ao lado do texto. Três detalhes importam:

- **`format`** precisa bater com o arquivo de verdade. O nosso é `.jpg`, então é `"jpeg"` — se estivéssemos usando um `.png`, teria que ser `"png"`.
- **`source.bytes`** recebe o **conteúdo** da imagem, não o caminho dela. É por isso que abrimos o arquivo em modo binário antes.
- A **imagem vem antes do texto** na lista `content`. Essa ordem é a recomendada: o modelo olha a imagem e depois lê o que pedimos sobre ela.

O modelo e a imagem viram argumentos com valor padrão, então basta chamar `analisar_imagem("...")`.

> **Detalhe do Python:** valores padrão são fixados no momento em que a função é **definida**. Se trocar o `MODEL_ID` ou a imagem depois, é preciso reexecutar a célula da função (ou passar os novos valores como argumento).

In [36]:
# Função para analisar a imagem utilizando o modelo de IA generativa do Amazon Bedrock.
# Recebe o prompt e, opcionalmente, o ID do modelo e os bytes da imagem.
def analisar_imagem(prompt, model_id=MODEL_ID, imagem=imagem_bytes):
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": "jpeg",
                            "source": {"bytes": imagem}
                        }
                    },
                    {"text": prompt}
                ]
            }
        ]
    )

    return response["output"]["message"]["content"][0]["text"]

### 9.4 Testando diferentes tarefas com a mesma imagem

Daqui em diante é a **mesma imagem** e a **mesma função** — só o prompt muda.

Cada uma das próximas células pede uma tarefa diferente: descrição detalhada, resumo em tópicos, insights e, por fim, extração do texto contido na imagem (o que na prática funciona como um OCR).

#### Descrição detalhada

In [37]:
# Testando a função analisar_imagem com um prompt para descrever a imagem.
print(
    analisar_imagem(
        "Descreva detalhadamente esta imagem."
    )
)

A imagem mostra um cérebro azul com uma mão robótica ao lado dele. A mão robótica tem um dedo apontando para o cérebro. A mão e o cérebro são coloridos em um gradiente de cores, incluindo azul, verde, amarelo e vermelho. A imagem é acompanhada por um logotipo da marca "Native". O fundo é um padrão de listras pretas e brancas com uma borda azul. A imagem retrata a interação entre humanos e robôs, destacando o papel da inteligência artificial e da tecnologia na compreensão do cérebro humano.


#### Resumo em tópicos

In [38]:
# Alterando o prompt para extrair outros tipos de informação
print(
    analisar_imagem(
        """
        Analise esta imagem e gere um resumo
        em até 5 tópicos.
        """
    )
)

Aqui está o resumo da imagem em até 5 tópicos:

1.  A imagem mostra um cérebro com uma mão de robô.
2.  A mão do robô tem um dedo indicador apontando para o cérebro.
3.  A palavra "native" está localizada no canto superior direito da imagem.
4.  A imagem tem um fundo preto com um design de circuito integrado.
5.  O cérebro tem uma cor de arco-íris.


#### Principais insights

In [39]:
# Alterando o prompt para extrair outros tipos de informação
print(
    analisar_imagem(
        """
        Analise esta imagem e apresente
        os principais insights encontrados.
        """
    )
)

A imagem representa uma mão humana tocando uma representação do cérebro humano, que é dividido em duas partes: uma em azul, representando a inteligência artificial, e outra em laranja, representando a inteligência humana. A imagem sugere uma colaboração entre a inteligência artificial e a inteligência humana. Além disso, a imagem tem um fundo escuro com um design abstrato, que pode representar a complexidade e a profundidade da inteligência artificial e da inteligência humana.


#### Extração de texto (OCR)

In [40]:
# Alterando o prompt para extrair outros tipos de informação
print(
    analisar_imagem(
        """
        Extraia o texto da imagem.
        """
    )
)

native 1001 1001


---
## 10. Resumo dos aprendizados

- **Boto3** é a biblioteca que conecta o Python à AWS. As credenciais ficam **fora do código** e todos os serviços saem de uma única `boto3.Session`.
- O Bedrock tem dois clientes: **`bedrock`** (catálogo, ex.: `list_foundation_models`) e **`bedrock-runtime`** (inferência, ex.: `invoke_model` e `converse`).
- Antes de usar um modelo, confira na listagem: `modelId`, `inputModalities`, `inferenceTypesSupported` (se só tiver `INFERENCE_PROFILE`, use o prefixo `us.`) e `modelLifecycle`. E lembre de liberar o acesso em `Bedrock → Model access`.
- **`max_tokens`** limita o tamanho (e o custo) da resposta; **`temperature`** e **`top_p`** controlam a variação. Ajuste um de cada vez.
- Regra prática: **criação** → temperatura alta; **explicação e extração de dados** → temperatura baixa.
- Na família Nova: **Micro** (só texto, mais rápido e barato) < **Lite** (multimodal, equilibrado) < **Pro** (multimodal, mais capaz e mais lento).
- **`invoke_model`** exige o JSON nativo de cada fabricante; **`converse`** padroniza a chamada — trocar de modelo vira trocar uma string.
- Para imagens: modelo com `IMAGE` em `inputModalities`, arquivo lido em bytes (`rb`), `format` correto e a imagem antes do texto em `content`.